In [193]:
import numpy as np
import plotly.graph_objects as go

# ---------------------------------------------------------------------------
# PARAMETERS & DATA DENSITY
# ---------------------------------------------------------------------------
density_control = 3
vector_scale = 1

# ---------------------------------------------------------------------------
# 1. Define a Randomized Cubic Surface (CONCAVE UP) & Analytic Gradients
# ---------------------------------------------------------------------------
np.random.seed(None)

c = {
    '0':   np.random.uniform(0.1, 0.5),
    'x':   np.random.uniform(-0.2, 0.2),
    'y':   np.random.uniform(-0.2, 0.2),
    'x2':  np.random.uniform(0.05, 0.25),
    'xy':  np.random.uniform(-0.1, 0.1),
    'y2':  np.random.uniform(0.05, 0.25),
    'x3':  np.random.uniform(-0.02, 0.02),
    'x2y': np.random.uniform(-0.02, 0.02),
    'xy2': np.random.uniform(-0.02, 0.02),
    'y3':  np.random.uniform(-0.02, 0.02)
}

def surface_z(x, y):
    return (c['0'] + c['x']*x + c['y']*y +
            c['x2'] * x**2 + c['xy'] * x * y + c['y2'] * y**2 +
            c['x3'] * x**3 + c['x2y'] * (x**2)*y + c['xy2'] * x*(y**2) + c['y3'] * y**3)

def surface_gradients(x, y):
    df_dx = (c['x'] + 2 * c['x2'] * x + c['xy'] * y +
             3 * c['x3'] * x**2 + 2 * c['x2y'] * x * y + c['xy2'] * y**2)
    df_dy = (c['y'] + c['xy'] * x + 2 * c['y2'] * y +
             c['x2y'] * x**2 + 2 * c['xy2'] * x * y + 3 * c['y3'] * y**2)
    return df_dx, df_dy

x_range = np.linspace(-2, 2, 50)
y_range = np.linspace(-2, 2, 50)
X, Y = np.meshgrid(x_range, y_range)
Z = surface_z(X, Y)

# ---------------------------------------------------------------------------
# 2. RANDOMIZE HYPERPLANES WITH OCCASIONAL APPROXIMATE OVERLAPS
# ---------------------------------------------------------------------------
theta1 = np.random.uniform(0, np.pi)

if np.random.rand() < 0.4:
    angle_delta = np.random.uniform(2 * np.pi / 180, 10 * np.pi / 180) * np.sign(np.random.uniform(-1, 1))
    theta2 = theta1 + angle_delta
    x_intersect = np.random.uniform(-0.4, 0.4)
    y_intersect = np.random.uniform(-0.4, 0.4)
else:
    theta2 = np.random.uniform(0, np.pi)
    while abs(np.sin(theta1 - theta2)) < 0.3:
        theta2 = np.random.uniform(0, np.pi)
    x_intersect = np.random.uniform(-0.8, 0.8)
    y_intersect = np.random.uniform(-0.8, 0.8)

A1, B1 = np.cos(theta1), np.sin(theta1)
A2, B2 = np.cos(theta2), np.sin(theta2)

D1 = A1 * x_intersect + B1 * y_intersect
D2 = A2 * x_intersect + B2 * y_intersect

# ---------------------------------------------------------------------------
# 3. Define Flow Field & Derivatives
# ---------------------------------------------------------------------------
freqs = np.random.uniform(0.5, 1.5, size=(4,))
phases = np.random.uniform(0, 2 * np.pi, size=(4,))
amps = np.random.uniform(0.5, 1.5, size=(4,))

def flow_field(x, y):
    vx_raw = amps[0] * np.sin(freqs[0] * y + phases[0]) + amps[1] * np.cos(freqs[1] * x + phases[1])
    vy_raw = amps[2] * np.sin(freqs[2] * x + phases[2]) + amps[3] * np.cos(freqs[3] * y + phases[3])
    df_dx, df_dy = surface_gradients(x, y)
    vz = df_dx * vx_raw + df_dy * vy_raw
    return vx_raw, vy_raw, vz

def calculate_derivatives_at(x, y):
    vx, vy, vz = flow_field(x, y)
    v_vec = np.array([vx, vy, vz])

    eps = 1e-5
    vx_xp, vy_xp, vz_xp = flow_field(x + eps, y)
    vx_xm, vy_xm, vz_xm = flow_field(x - eps, y)
    dv_dx = np.array([vx_xp - vx_xm, vy_xp - vy_xm, vz_xp - vz_xm]) / (2 * eps)

    vx_yp, vy_yp, vz_yp = flow_field(x, y + eps)
    vx_ym, vy_ym, vz_ym = flow_field(x, y - eps)
    dv_dy = np.array([vx_yp - vx_ym, vy_yp - vy_ym, vz_yp - vz_ym]) / (2 * eps)

    accel_vec = vx * dv_dx + vy * dv_dy
    return v_vec, accel_vec

# ---------------------------------------------------------------------------
# 4. RK4 Streamline Integrator (Decoupled initialization)
# ---------------------------------------------------------------------------
def integrate_half_streamline(start_xy, steps=450, dt=0.015, forward=True):
    direction = 1.0 if forward else -1.0
    h = dt * direction
    pts = [np.array([start_xy[0], start_xy[1], surface_z(start_xy[0], start_xy[1])])]

    for _ in range(steps):
        curr = pts[-1]
        x, y, _ = curr

        k1_vx, k1_vy, _ = flow_field(x, y)
        ax, ay = x + 0.5 * h * k1_vx, y + 0.5 * h * k1_vy
        k2_vx, k2_vy, _ = flow_field(ax, ay)
        bx, by = x + 0.5 * h * k2_vx, y + 0.5 * h * k2_vy
        k3_vx, k3_vy, _ = flow_field(bx, by)
        cx, cy = x + h * k3_vx, y + h * k3_vy
        k4_vx, k4_vy, _ = flow_field(cx, cy)

        nxt_x = x + (h / 6.0) * (k1_vx + 2*k2_vx + 2*k3_vx + k4_vx)
        nxt_y = y + (h / 6.0) * (k1_vy + 2*k2_vy + 2*k3_vy + k4_vy)

        if not (-2.0 <= nxt_x <= 2.0 and -2.0 <= nxt_y <= 2.0):
            break

        nxt_z = surface_z(nxt_x, nxt_y)
        pts.append(np.array([nxt_x, nxt_y, nxt_z]))

    return np.array(pts)

def generate_boundary_crossing_streamline():
    # MODIFIED: Instead of starting at the intersection, start at a random point
    # along Plane 1 line equation, shifted away from the intersection line point.
    t_offset = np.random.uniform(-1.2, 1.2)
    # Ensure we don't start right on the intersection point by accident
    if abs(t_offset) < 0.2:
        t_offset += 0.4 * np.sign(t_offset if t_offset != 0 else 1)

    start_x = x_intersect - B1 * t_offset
    start_y = y_intersect + A1 * t_offset

    # Clip starting parameters to domain bounds
    start_xy = np.array([np.clip(start_x, -1.5, 1.5), np.clip(start_y, -1.5, 1.5)])

    forward_pts = integrate_half_streamline(start_xy, forward=True)
    backward_pts = integrate_half_streamline(start_xy, forward=False)
    return np.vstack((np.flip(backward_pts, axis=0)[:-1], forward_pts))

streamline = generate_boundary_crossing_streamline()

# ---------------------------------------------------------------------------
# 5. Locate Exact Boundary Crossings
# ---------------------------------------------------------------------------
def get_implicit_crossing_point(pts, A, B, D):
    for i in range(len(pts) - 1):
        val1 = A * pts[i][0] + B * pts[i][1] - D
        val2 = A * pts[i+1][0] + B * pts[i+1][1] - D
        if val1 * val2 <= 0:
            t = abs(val1) / (abs(val1) + abs(val2))
            return pts[i][:2] + t * (pts[i+1][:2] - pts[i][:2])
    return None

cross1_xy = get_implicit_crossing_point(streamline, A1, B1, D1)
cross2_xy = get_implicit_crossing_point(streamline, A2, B2, D2)

# ---------------------------------------------------------------------------
# 6. Build Interactive Plotly Figure
# ---------------------------------------------------------------------------
fig = go.Figure()

# --- A. Plot Base Cubic Surface ---
gray_colorscale = [[0, 'rgb(215, 215, 215)'], [1, 'rgb(215, 215, 215)']]
fig.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale=gray_colorscale, showscale=False, opacity=0.55, name="Cubic Surface"))

# --- B. Generate Dynamic Parametric Meshes for Hyperplanes ---
z_min, z_max = np.min(Z) - 0.2, np.max(Z) + 0.2
z_plane_range = np.linspace(z_min, z_max, 10)
t_line = np.linspace(-2.5, 2.5, 50)

# Plane 1 Geometry
p1_x_line = (D1 * A1) - B1 * t_line
p1_y_line = (D1 * B1) + A1 * t_line
P1_X, P1_Z = np.meshgrid(p1_x_line, z_plane_range)
P1_Y, _ = np.meshgrid(p1_y_line, z_plane_range)

# Plane 2 Geometry
p2_x_line = (D2 * A2) - B2 * t_line
p2_y_line = (D2 * B2) + A2 * t_line
P2_X, P2_Z = np.meshgrid(p2_x_line, z_plane_range)
P2_Y, _ = np.meshgrid(p2_y_line, z_plane_range)

# Surface Intersection Curves
t_curve = np.linspace(-2.0, 2.0, 200)
p1_cx = np.clip((D1 * A1) - B1 * t_curve, -2, 2)
p1_cy = np.clip((D1 * B1) + A1 * t_curve, -2, 2)
p2_cx = np.clip((D2 * A2) - B2 * t_curve, -2, 2)
p2_cy = np.clip((D2 * B2) + A2 * t_curve, -2, 2)

# Plot Plane 1 (Red)
fig.add_trace(go.Surface(x=P1_X, y=P1_Y, z=P1_Z, colorscale=[[0, 'rgba(255,0,0,0.12)'], [1, 'rgba(255,0,0,0.12)']], showscale=False, opacity=0.25, hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter3d(x=p1_cx, y=p1_cy, z=surface_z(p1_cx, p1_cy), mode='lines', line=dict(color='red', width=5), name="Intersection Plane 1"))

# Plot Plane 2 (Orange)
fig.add_trace(go.Surface(x=P2_X, y=P2_Y, z=P2_Z, colorscale=[[0, 'rgba(255,165,0,0.12)'], [1, 'rgba(255,165,0,0.12)']], showscale=False, opacity=0.25, hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter3d(x=p2_cx, y=p2_cy, z=surface_z(p2_cx, p2_cy), mode='lines', line=dict(color='orange', width=5), name="Intersection Plane 2"))

# --- C. Plot Single Black Streamline ---
fig.add_trace(go.Scatter3d(x=streamline[:, 0], y=streamline[:, 1], z=streamline[:, 2], mode='lines', line=dict(color='black', width=7), name="Spanning Streamline"))

# --- D. Background Flow Field Cones ---
X_q, Y_q = X[::density_control, ::density_control], Y[::density_control, ::density_control]
Vx, Vy, Vz = flow_field(X_q, Y_q)
fig.add_trace(go.Cone(x=X_q.flatten(), y=Y_q.flatten(), z=surface_z(X_q, Y_q).flatten(), u=Vx.flatten(), v=Vy.flatten(), w=Vz.flatten(), colorscale=[[0, 'rgb(120,120,120)'], [1, 'rgb(120,120,120)']], showscale=False, sizemode="scaled", sizeref=0.18, name="Flow Field"))

# --- E. Shaft + Cone Vector Head Drawing Function ---
def add_arrow_with_shaft(fig, start_pt, direction_vec, color, name, scale_factor):
    mag = np.linalg.norm(direction_vec)
    if mag == 0: return
    dir_norm = direction_vec / mag
    scaled_vec = dir_norm * scale_factor
    end_pt = start_pt + scaled_vec

    fig.add_trace(go.Scatter3d(
        x=[start_pt[0], end_pt[0]], y=[start_pt[1], end_pt[1]], z=[start_pt[2], end_pt[2]],
        mode='lines', line=dict(color=color, width=6), name=name, legendgroup=name
    ))
    fig.add_trace(go.Cone(
        x=[end_pt[0]], y=[end_pt[1]], z=[end_pt[2]], u=[dir_norm[0]], v=[dir_norm[1]], w=[dir_norm[2]],
        colorscale=[[0, color], [1, color]], showscale=False, sizemode="absolute", sizeref=0.15, name=name, legendgroup=name, showlegend=False
    ))

# Plot Derivative Pairs at Crossing Locations (if crossings exist)
for idx, (cross_xy, p_label) in enumerate(zip([cross1_xy, cross2_xy], ["P1", "P2"])):
    if cross_xy is not None:
        p_z = surface_z(cross_xy[0], cross_xy[1])
        pos_3d = np.array([cross_xy[0], cross_xy[1], p_z])
        v_vector, a_vector = calculate_derivatives_at(cross_xy[0], cross_xy[1])

        # Tangent Vector (Cyan/Blue)
      #  add_arrow_with_shaft(fig, pos_3d, v_vector, 'rgb(0, 180, 216)', f"{p_label} Tangent (v)", scale_factor=vector_scale)
        # Second Directional Derivative Vector (Purple)
        add_arrow_with_shaft(fig, pos_3d, a_vector, 'rgb(114, 9, 183)', f"{p_label} 2nd Dir Deriv (vᵀ H v)", scale_factor=vector_scale)

        #fig.add_trace(go.Scatter3d(x=[pos_3d[0]], y=[pos_3d[1]], z=[pos_3d[2]], mode='markers', marker=dict(color='black', size=5), showlegend=False))

# --- F. Layout Configuration ---
hidden_axis = dict(visible=False)
fig.update_layout(
    title="Decoupled Streamline Trajectories over Cubic Surface Manifold",
    scene=dict(xaxis=hidden_axis, yaxis=hidden_axis, zaxis=hidden_axis, aspectmode='cube'),
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.95)
)

fig.show()